In [1]:
import numpy as np
from farm.plant import Plant
from farm import Climate
from farm import Soil
from farm import Crop
from farm import CropModel
from farm.functions import evolved_calc_yield, plot_lin_regression
import pandas as pd
from pandas import DataFrame
from numpy import zeros
import matplotlib.pyplot as plt

/Users/cellaschnabel/miniforge3/envs/maize-Toff/lib/python3.10/site-packages/proplot/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as pkg


In [2]:
# ── Publication style defaults ──────────────────────────────────────────────
# Run this cell before any figure cell.
# Sets global rcParams (font sizes, legend, savefig DPI) and two helpers:
#   pub_label(ax, "(a)") — bold panel label at top-left
#   despine(ax)          — remove top and right spines
import matplotlib as mpl
import os; os.makedirs("output", exist_ok=True)

mpl.rcParams.update({
    "font.size":       10,
    "axes.labelsize":  11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "legend.frameon":  False,
    "savefig.dpi":     300,
    "savefig.bbox":    "tight",
})

def pub_label(ax, label, x=-0.08, y=1.02):
    """Bold panel label — (a), (b), … — at top-left of an axes."""
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=12, fontweight="bold", va="bottom", ha="left")

def despine(ax):
    """Remove top and right spines (journal standard)."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

In [3]:
# Fit Y_MAX regression from Kenya Seed Co. data (yield in kg/acre vs. days-to-maturity).
# Passing m and b explicitly avoids the internal /1000 unit issue in evolved_calc_yield.
_yd = pd.read_csv('data/Yields/hybrid_yields_verified.csv')
_, _m_yield, _b_yield = plot_lin_regression(
    _yd.verified_days_to_maturity,
    _yd.verified_yield_kg_acre,
    'verified_days_to_maturity', 'verified_yield_kg_acre', _yd,
    positive=False
)

## Model definitions

In [4]:
class Tree(Plant):
    """
    Simple tree component for agroforestry modeling.

    canopy_cover  : fraction of plot area covered by tree canopy [0-1].
                    Scales all plot-level tree fluxes (T, shade, HR).
                    Typical parkland agroforestry: 0.10 - 0.30.
    leaf_fraction : fraction of the 180-day crop season the tree carries leaves [0-1].
                    0.0 → leafless entire crop season (Faidherbia albida style:
                          no competition, no shade during crop season; HR still
                          operates via deep roots).
                    0.1 → leafed for first ~18 days, then dormant (~1-month overlap
                          before Faidherbia sheds leaves, per Roupsard 1999).
                    1.0 → leafed entire crop season (evergreen co-season tree).
                    Outside the crop season the tree is always active (kc=1.0).
                    Ref: Roupsard et al. (1999), Bayala & Prieto (2020).
    """
    def __init__(self, kc=1.0, LAI_max=2.0, canopy_cover=0.2,
                 leaf_fraction=1.0, *args, **kwargs):
        self.kc = kc
        self.LAI_max = LAI_max
        self.canopy_cover = canopy_cover
        self.leaf_fraction = leaf_fraction
        super().__init__(*args, **kwargs)

    def calc_LAI(self):
        return self.LAI_max

    def calc_I(self, LAI=None, int_efficiency=0.5):
        LAI_use = self.calc_LAI() if LAI is None else LAI
        return LAI_use * int_efficiency

    def calc_T(self, s, LAI=None, kc=None):
        kc_use = self.kc if kc is None else kc
        if s >= self.s_star:
            return kc_use * self.T_MAX
        elif s >= self.sw:
            return (s - self.sw) / (self.s_star - self.sw) * (kc_use * self.T_MAX)
        else:
            return 0.0


In [ ]:
def shade_factor(LAI_tree, canopy_cover=1.0, k_shade=0.35):
    """
    Plot-average fraction of incoming radiation (and thus PET) remaining after tree shading.

    Spatial weighted average: under-canopy fraction (cc) attenuated by Beer-Lambert
    extinction, plus open-sky fraction (1-cc) at full radiation:

        sf = cc * exp(-k_s * LAI_tree) + (1 - cc)

    Caylor et al. (2005): k_s = 0.35 for savanna parkland.
    Applied to both E and T_crop via Priestley-Taylor: PET proportional to available energy Q.

    LAI_tree    : tree leaf area index (stand-level, NOT pre-multiplied by canopy_cover)
    canopy_cover: fraction of plot area under tree canopy [0-1]
    k_shade     : extinction coefficient [-]; 0.35 from Caylor et al. 2005

    Returns value in (0, 1].
    """
    return canopy_cover * np.exp(-k_shade * LAI_tree) + (1.0 - canopy_cover)


In [ ]:
class TreeCropModel:
    def __init__(self, soil=None, climate=None, crop=None, tree=None, hr_max=1.0):
        """
        hr_max : plot-level hydraulic redistribution capacity [mm/day].
                 NOT scaled by canopy_cover — hr_max is already a plot-level rate.
                 HR flux is proportional to the water-potential gradient (Clapp-Hornberger),
                 normalised so hr_max is the flux when s_deep ≈ s_fc and s_top ≈ crop sw.
                 Literature: 0.1-1.0 mm/night at plot scale in savanna parkland
                 (Burgess et al. 2001, Caldwell et al. 1991, Bayala & Prieto 2020).
        """
        self.soil = soil
        self.crop = crop
        self.tree = tree
        self.climate = climate
        self.hr_max = hr_max

        self.nZr_top = soil.set_nZr(self.crop)

        class _Tmp: pass
        tmp = _Tmp()
        tmp.Zr = max(self.tree.Zr - self.crop.Zr, 0.0)
        self.nZr_deep_full = soil.set_nZr(tmp) if tmp.Zr > 0 else 0.0

        # Reference Δψ for HR normalisation [MPa]:
        # ψ at field capacity (deep, well-watered) minus ψ at crop wilting point (top, dry).
        # Ensures hr_max [mm/day] is the flux when s_deep ≈ s_fc and s_top ≈ crop sw.
        psi_fc = soil.Psi_S_MPa * soil.sfc ** (-soil.b)
        psi_sw = soil.Psi_S_MPa * self.crop.sw ** (-soil.b)
        self._delta_psi_ref = abs(psi_sw - psi_fc)  # MPa > 0

    def _calc_L(self, s0, nZr):
        """Laio (2001) gravity drainage for a soil layer with given nZr [mm/day].
        Same physics as soil.calc_L() but accepts nZr directly rather than requiring
        soil.nZr to be set, so it works correctly for both the top and deep layers.
        """
        Lmax = s0 - self.soil.sfc
        if Lmax > 0 and nZr > 0:
            beta = 2 * self.soil.b + 4
            m = self.soil.Ks / (nZr * (np.exp(beta * (1 - self.soil.sfc)) - 1))
            L = (1 / beta) * np.log(
                np.exp(beta * (s0 - self.soil.sfc))
                - np.exp(-m * beta) * (np.exp(beta * (s0 - self.soil.sfc)) - 1))
            return min(L, Lmax) * nZr
        return 0.0

    def pre_allocate(self):
        n = self.n_days
        self.R          = zeros(n)
        self.s_top      = zeros(n + 1)
        self.s_deep     = zeros(n + 1)
        self.ET         = zeros(n)
        self.I          = zeros(n)
        self.E          = zeros(n)
        self.T          = zeros(n)
        self.L          = zeros(n)
        self.Q          = zeros(n)
        self.HR         = zeros(n)   # hydraulic redistribution flux [mm/day]
        self.kc_tree    = zeros(n)   # effective tree kc (0 when dormant, i.e. past leaf_fraction×lgp days)
        self.dos        = zeros(n)
        self.dsdt       = zeros(n)
        self.LAI        = zeros(n)
        self.kc         = zeros(n)
        self.stress     = zeros(n)
        self.T_tree     = zeros(n)
        self.ET_crop    = zeros(n)
        self.ET_tree    = zeros(n)
        self.T_tree_top = zeros(n)
        self.T_tree_deep= zeros(n)

    def run(self, do_output=False, s0=0.3, planting_date=100,
            t_before=60, t_after=7,
            deep_roots=True, shade_on=True, hr_on=True, ci_on=True,
            s_deep_0=None):
        """
        deep_roots : trees access the deep layer (default True)
        shade_on   : tree canopy reduces PET under canopy (default True).
                     Shade attenuates incoming radiation (Beer-Lambert, k=0.35;
                     Caylor et al. 2005), reducing available energy Q for all ET.
                     Applied to both E and T_crop (Priestley-Taylor: PET ∝ Q).
        hr_on      : hydraulic redistribution active (default True).
                     Trees passively lift water from deep to top when s_deep > s_top.
        ci_on      : tree canopy interception active (default True).
                     When False, only crop interception is computed.
        """
        self.planting_date = planting_date
        self.n_days = t_before + self.crop.lgp + t_after
        self.pre_allocate()
        self.nZr_deep = self.nZr_deep_full if deep_roots else 0.0

        self.doy_start = self.planting_date - t_before
        if self.doy_start <= 0:
            self.doy_start = 365 + self.doy_start

        doy = np.arange(self.doy_start, self.doy_start + self.n_days)
        while (doy - 365 > 0).any():
            doy = doy - 365 * ((doy - 365) > 0)

        for t in range(self.n_days):
            self.R[t] = self.climate.rainfall[doy[t] - 1]
        self.doy = doy

        # Both layers start at s0: no artificial deep-layer wetness at initialisation.
        # Pre-season Laikipia vadose zone is depleted, not near field capacity.
        # Use spin_up() or pass explicit s_deep_0 when a specific initial condition is needed.
        self.s_top[0]  = s0
        self.s_deep[0] = s_deep_0 if s_deep_0 is not None else s0
        dos = 0
        planted = False

        for t in range(self.n_days):
            if self.doy[t] == self.planting_date:
                planted = True
            if planted:
                dos += 1

            self.dos[t] = dos
            self.kc[t]  = self.crop.calc_kc(dos)
            self.LAI[t] = self.crop.calc_LAI(self.kc[t])
            self.stress[t] = self.crop.calc_stress(self.s_top[t])  # beginning-of-day, matches CropModel

            # --- Tree phenology: kc and LAI for this timestep
            # leaf_fraction controls what fraction of the crop season the tree carries leaves.
            # 0.0 → dormant entire crop season (Faidherbia albida style).
            # 1.0 → leafed entire crop season (evergreen co-season tree).
            # Outside the crop season the tree is always active (kc=1.0).
            in_crop_season = planted and 0 < dos <= self.crop.lgp
            leaf_day_limit = self.tree.leaf_fraction * self.crop.lgp
            if in_crop_season and dos > leaf_day_limit:
                tree_kc  = 0.0   # dormant: past leaf-carrying window this season
                tree_LAI = 0.0
            else:
                tree_kc  = self.tree.kc
                tree_LAI = self.tree.calc_LAI()
            self.kc_tree[t] = tree_kc

            self.Q[t] = self.soil.calc_Q(self.s_top[t], units="mm/day")

            # --- Interception: crop + tree canopy (Fix 5)
            # Tree interception is scaled by canopy_cover (plot-level fraction).
            # For leaf_fraction=0.0, tree_LAI=0 during crop season → no tree interception.
            # ci_on=False isolates tree interception as a mechanism (for experiments).
            I_crop = self.crop.calc_I(self.LAI[t])
            I_tree = self.tree.calc_I(LAI=tree_LAI) * self.tree.canopy_cover if ci_on else 0.0
            self.I[t] = min(I_crop + I_tree, self.R[t])
            rainfall  = max(self.R[t] - self.I[t], 0.0)

            s_top  = self.s_top[t]
            s_deep = self.s_deep[t]

            # --- Rainfall input to top layer
            s_top += (rainfall - self.Q[t]) / self.nZr_top

            # --- Top-layer drainage (Fix 3 extended)
            # Same Laio (2001) physics as the published CropModel single-layer drainage.
            # Routes to deep layer if present; exits as L if nZr_deep=0 (deep_roots=False).
            # This ensures monoculture equivalence: TreeCropModel with a ghost tree
            # (deep_roots=False, T_MAX=0, canopy_cover=0) matches CropModel exactly.
            if s_top > self.soil.sfc:
                perc_mm = self._calc_L(s_top, self.nZr_top)
                s_top  -= perc_mm / self.nZr_top
                if self.nZr_deep > 0:
                    s_deep += perc_mm / self.nZr_deep
                else:
                    self.L[t] += perc_mm

            # --- Saturation overflow
            if s_top > 1.0:
                excess_mm = (s_top - 1.0) * self.nZr_top
                s_top = 1.0
                if self.nZr_deep > 0:
                    s_deep += excess_mm / self.nZr_deep
                else:
                    self.L[t] += excess_mm

            # --- Hydraulic redistribution: deep -> top driven by Δψ (Fix 1)
            # Flux is proportional to the Clapp-Hornberger water-potential gradient,
            # normalised by Δψ_ref (s_fc → sw) so hr_max [mm/day] is the flux at the
            # maximum agronomically relevant gradient. Operates regardless of leaf_fraction.
            # Ref: Caldwell et al. (1991), Burgess et al. (2001), Bayala & Prieto (2020)
            if hr_on and self.nZr_deep > 0 and s_deep > s_top:
                psi_top  = self.soil.Psi_S_MPa * max(s_top,  1e-6) ** (-self.soil.b)
                psi_deep = self.soil.Psi_S_MPa * max(s_deep, 1e-6) ** (-self.soil.b)
                delta_psi = max(psi_deep - psi_top, 0.0)  # MPa, positive when deep wetter
                hr_flux = min(
                    self.hr_max * delta_psi / self._delta_psi_ref,
                    s_deep * self.nZr_deep,
                    (1.0 - s_top) * self.nZr_top
                )
                s_top  += hr_flux / self.nZr_top
                s_deep -= hr_flux / self.nZr_deep
                self.HR[t] = hr_flux

            # --- Tree transpiration (scaled by canopy_cover; uses leaf_fraction-aware kc)
            T_tree_canopy_pot  = self.tree.calc_T(1.0,              kc=tree_kc) * self.tree.canopy_cover
            T_tree_top_supply  = self.tree.calc_T(max(s_top,  0.0), kc=tree_kc) * self.tree.canopy_cover
            T_tree_deep_supply = self.tree.calc_T(max(s_deep, 0.0), kc=tree_kc) * self.tree.canopy_cover
            T_tree_soil_supply = T_tree_top_supply + T_tree_deep_supply

            T_tree_total_pot = min(T_tree_canopy_pot, T_tree_soil_supply)

            # Deep-first: tree preferentially uses deep roots, falling back to
            # shallow only when deep supply is exhausted. Ecophysiologically
            # correct for Faidherbia — its taproot system evolved specifically
            # to access deep water unavailable to crops.
            T_tree_deep_pot = min(T_tree_total_pot, T_tree_deep_supply)
            T_tree_top_pot  = max(T_tree_total_pot - T_tree_deep_pot, 0.0)

            # --- Crop + soil evaporation (microclimate: shade reduces available radiation)
            # Beer-Lambert attenuation of incoming shortwave (k=0.35; Caylor et al. 2005)
            # reduces available energy Q under the canopy. In a Priestley-Taylor framework,
            # PET ∝ Q, so shade lowers atmospheric demand for both E and T_crop.
            s_eff = max(s_top, 0.0)
            T_crop_pot = self.crop.calc_T(s_eff, LAI=self.LAI[t])
            E_pot      = self.climate.calc_E(s_eff, LAI=self.LAI[t], sh=self.soil.sh)
            sf = shade_factor(tree_LAI, self.tree.canopy_cover) if shade_on else 1.0
            E_pot      *= sf
            T_crop_pot *= sf

            # --- Top-layer allocation (ration when demand > supply)
            demand_top    = T_tree_top_pot + T_crop_pot + E_pot
            available_top = s_top * self.nZr_top
            if demand_top <= available_top:
                T_tree_shallow = T_tree_top_pot
                T_crop_actual  = T_crop_pot
                E_actual       = E_pot
            else:
                scale = available_top / demand_top if demand_top > 0 else 0.0
                T_tree_shallow = T_tree_top_pot * scale
                T_crop_actual  = T_crop_pot     * scale
                E_actual       = E_pot          * scale
            s_top -= (T_tree_shallow + T_crop_actual + E_actual) / self.nZr_top

            # --- Deep-layer tree uptake (backfill if shallow was rationed)
            # If top-layer rationing reduced tree's shallow uptake, compensate from deep,
            # capped by deep moisture supply and total canopy potential.
            T_tree_deep_pot = min(T_tree_total_pot - T_tree_shallow, T_tree_deep_supply)
            T_tree_deep = 0.0
            if T_tree_deep_pot > 0 and self.nZr_deep > 0:
                T_tree_deep = min(T_tree_deep_pot, s_deep * self.nZr_deep)
                s_deep -= T_tree_deep / self.nZr_deep

            T_tree_actual = T_tree_shallow + T_tree_deep

            # --- Save fluxes
            self.T_tree[t]      = T_tree_actual
            self.T[t]           = T_crop_actual
            self.E[t]           = E_actual
            self.ET_crop[t]     = T_crop_actual + E_actual
            self.ET_tree[t]     = T_tree_actual
            self.ET[t]          = self.ET_crop[t] + self.ET_tree[t]
            self.T_tree_top[t]  = T_tree_shallow
            self.T_tree_deep[t] = T_tree_deep

            # --- Deep layer gravity drainage (Fix 2: Laio 2001, same physics as top layer)
            # Water above field capacity in the deep layer drains out of the system.
            # Without this, deep water is unrealistically permanent once it percolates.
            if self.nZr_deep > 0:
                L_deep = self._calc_L(s_deep, self.nZr_deep)
                s_deep -= L_deep / self.nZr_deep
                self.L[t] += L_deep

            # --- Deep saturation overflow
            if s_deep > 1.0:
                self.L[t] += (s_deep - 1.0) * self.nZr_deep
                s_deep = 1.0

            self.s_top[t+1]  = max(s_top,  0.0)
            self.s_deep[t+1] = max(s_deep, 0.0)
            total_now  = self.s_top[t]   * self.nZr_top + self.s_deep[t]   * self.nZr_deep
            total_next = self.s_top[t+1] * self.nZr_top + self.s_deep[t+1] * self.nZr_deep
            self.dsdt[t] = total_next - total_now

        if do_output:
            return self.output()

    def spin_up(self, n_seasons=5, s0=0.3, s_deep_0=None, **run_kwargs):
        """
        Run N silent seasons to converge initial conditions.
        Chains end-of-season s_top → next s0, s_deep → next s_deep_0.
        Returns (s_top_converged, s_deep_converged) as scalars.

        n_seasons=5 is sufficient for deep-layer convergence under typical
        Laikipia climatologies (convergence usually within 2-3 seasons).
        The same stochastic rainfall sequence is reused each spin-up season,
        which is acceptable because the goal is vadose-zone convergence, not
        full interannual variability during spin-up.
        """
        s_t = s0
        s_d = s_deep_0 if s_deep_0 is not None else s0
        for _ in range(n_seasons):
            self.run(s0=s_t, s_deep_0=s_d, **run_kwargs)
            out = self.output()
            s_t = float(out["s_top"].iloc[-1])
            s_d = float(out["s_deep"].iloc[-1])
        return s_t, s_d

    def output(self):
        return DataFrame({
            "kc":         self.kc,
            "kc_tree":    self.kc_tree,
            "LAI":        self.LAI,
            "stress":     self.stress,
            "R":          self.R,
            "s_top":      self.s_top[:-1],
            "s_deep":     self.s_deep[:-1],
            "I":          self.I,
            "Q":          self.Q,
            "E":          self.E,
            "T_crop":     self.T,
            "T_tree":     self.T_tree,
            "ET_crop":    self.ET_crop,
            "ET_tree":    self.ET_tree,
            "ET":         self.ET,
            "HR":         self.HR,
            "L":          self.L,
            "dsdt":       self.dsdt,
            "dos":        self.dos,
            "doy":        self.doy,
            "T_tree_top": self.T_tree_top,
            "T_tree_deep":self.T_tree_deep,
        })


In [ ]:
def run_one_season(model_cls, soil_cls, crop_cls, climate_obj, *,
                   texture="loam", lgp=180,
                   planting_date=100, t_before=60, t_after=7, s0=0.3,
                   n_spinup=0, t_before_spinup=None,
                   tree_params=None):

    soil = soil_cls(texture=texture)
    crop = crop_cls(soil=soil)
    crop.lgp = lgp

    if tree_params is None:
        soil.set_nZr(crop)
        model = model_cls(crop=crop, climate=climate_obj, soil=soil)
    else:
        tree = Tree(soil=soil, **tree_params)
        model = model_cls(crop=crop, tree=tree, climate=climate_obj, soil=soil)

    # Spin-up: converge initial conditions before the real run.
    # Skipped for CropModel (no spin_up method) and when n_spinup=0.
    run_kwargs = dict(planting_date=planting_date, t_before=t_before, t_after=t_after)
    if n_spinup > 0 and hasattr(model, "spin_up"):
        spinup_kwargs = run_kwargs.copy()
        if t_before_spinup is not None:
            spinup_kwargs["t_before"] = t_before_spinup
        s0, s_deep_0 = model.spin_up(n_seasons=n_spinup, s0=s0, **spinup_kwargs)
        run_kwargs["s_deep_0"] = s_deep_0

    model.run(s0=s0, **run_kwargs)
    return crop, model.output()


def season_metrics(crop, df, lgp, sf=1.0):
    s_use = df["s_top"].values if "s_top" in df.columns else df["s"].values
    y_max = evolved_calc_yield(dtm=lgp, m=_m_yield, b=_b_yield)

    avg_stress, dyn_stress, yld = crop.calc_dstress(
        s=s_use, stress=df["stress"].values, Y_MAX=y_max
    )
    return avg_stress, dyn_stress, yld, df["R"].sum()


def _tree_sf(tree_params):
    """Crop-season shade factor for a given tree parameter dict.
    Scales by leaf_fraction (fraction of crop season the tree is leafed).
    leaf_fraction=0.0 → no shade during crop season → sf=1.0.
    leaf_fraction=1.0 → full-season shade → shade_factor(LAI_max, canopy_cover).
    """
    lf = tree_params.get("leaf_fraction", 1.0)
    if lf == 0.0:
        return 1.0   # fully dormant during crop season → no shade penalty
    cc  = tree_params.get("canopy_cover", 0.2)
    lai = tree_params.get("LAI_max", 2.0)
    sf_leafed = shade_factor(lai, cc)
    # Season-average: lf fraction leafed (shade), rest bare (sf=1)
    return lf * sf_leafed + (1.0 - lf)


def simulate_systems(n_sims, *, lgp=180, texture="loam", seed=123,
                     tree_params=None, n_spinup=5):
    rng = np.random.default_rng(seed)
    rows_mono, rows_tree = [], []
    outputs_mono, outputs_tree = [], []

    if tree_params is None:
        tree_params = dict(
            Zr=1500,
            T_MAX=1.0,
            sw_MPa=-4.0,
            s_star_MPa=-0.1,
            kc=1.0,
            canopy_cover=0.2,
            leaf_fraction=0.0,
        )

    sf_tree = _tree_sf(tree_params)

    for i in range(n_sims):
        np.random.seed(rng.integers(0, 2**32 - 1, dtype=np.uint32).item())
        climate = Climate()

        crop_m, out_m = run_one_season(CropModel, Soil, Crop, climate,
                                       texture=texture, lgp=lgp,
                                       n_spinup=n_spinup)
        a, d, y, rf = season_metrics(crop_m, out_m, lgp)
        rows_mono.append(dict(Sim=i, AvgStress=a, DynamicStress=d, Yield=y, RF=rf))
        outputs_mono.append(out_m)

        crop_t, out_t = run_one_season(TreeCropModel, Soil, Crop, climate,
                                       texture=texture, lgp=lgp,
                                       n_spinup=n_spinup,
                                       tree_params=tree_params)
        a, d, y, rf = season_metrics(crop_t, out_t, lgp, sf=sf_tree)
        rows_tree.append(dict(Sim=i, AvgStress=a, DynamicStress=d, Yield=y, RF=rf))
        outputs_tree.append(out_t)